# Imports

In [1]:
import os
import torch
import numpy as np
import open3d as o3d
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
from scene.dataset_readers import sceneLoadTypeCallbacks
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False

def prepare_output_and_logger(args):    
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])
        
    # Set up output folder
    print("Output folder: {}".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    # Create Tensorboard writer
    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# train.py
If you need more specific code review look at [tutorial_train_py.ipynb](tutorial_train_py.ipynb)

This tutorial follows the order in which the program starts after typing  
`python train.py -s your/file`  
on bash

## Parser Setting

In [2]:
parser = ArgumentParser(description="Training script parameters")
lp = ModelParams(parser)
op = OptimizationParams(parser)
pp = PipelineParams(parser)
parser.add_argument('--ip', type=str, default="127.0.0.1")
parser.add_argument('--port', type=int, default=6009)
parser.add_argument('--debug_from', type=int, default=-1)
parser.add_argument('--detect_anomaly', action='store_true', default=False)
parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--quiet", action="store_true")
parser.add_argument('--disable_viewer', action='store_true', default=False)
parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
parser.add_argument("--start_checkpoint", type=str, default=None)

# In Jupyter, parse_known_args avoids runtime arguments such as -f that are injected by the notebook kernel.
args, _ = parser.parse_known_args([])
args.save_iterations.append(args.iterations)

# Convert the parser namespace into the lightweight argument objects used by the training code.
model_args = lp.extract(args)
opt_args = op.extract(args)
pipe_args = pp.extract(args)
model_args.source_path = os.path.join(model_args.source_path,"GaussianTest/Test2") 
# source_path is hardcoded on purpose for this tutorial, but you can change it to your own dataset path.

print("Loaded parser-backed arguments")
print("sh_degree:", model_args.sh_degree)
print("optimizer_type:", opt_args.optimizer_type)
print("source path: ", model_args.source_path)


Loaded parser-backed arguments
sh_degree: 3
optimizer_type: default
source path:  c:\Dev\gaussian-splatting-for-practice\GaussianTest/Test2


## training method initialize
After parser is set, inside train.py training method is called, and the process below activates

In [3]:
first_iter = 0

When you use `prepare_output_and_logger` method, it creates new ouput folder with unique_str so you should check your output folder everytime after you run the code

In [4]:
tb_writer = prepare_output_and_logger(model_args)

Output folder: ./output/4a4af6ac-5
Tensorboard not available: not logging progress


In [5]:
gaussian = GaussianModel(model_args.sh_degree, opt_args.optimizer_type)
scene = Scene(model_args, gaussian)
print("GaussianModel initialized successfully")

Reading camera 25/25
Loading Training Cameras


c:\Users\COM\anaconda3\envs\gaussian_splatting\lib\site-packages\torch\cuda\__init__.py:218: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Loading Test Cameras
Number of points at initialisation :  1768
GaussianModel initialized successfully


Loading Test Cameras # Scene.init.py Line 72  
Number of points at initialisation :  1768 # scene.gaussian_model.py Line 157 (create_from_pcd)

After you run the code above it creates `input.ply` on output folder from `prepare_and_logger` which has the same data from [GaussianTest/Test2/sparse/0](GaussianTest/Test2/sparse/0)/points3D.ply 

### Comparing input.ply and point3D.ply
If you need more specific code review look at [tutorial_train_py.ipynb](tutorial_train_py.ipynb) `Chapter Input`

In [18]:
ply_path = os.path.join(model_args.model_path, "input.ply")
ply_path2 = os.path.join(model_args.source_path, "sparse/0/points3D.ply")
pcd = o3d.io.read_point_cloud(ply_path)
pcd2= o3d.io.read_point_cloud(ply_path2)

pcd.paint_uniform_color([1.0, 0.0, 0.0]) # Red
pcd2.paint_uniform_color([0.0, 0.0, 1.0]) # Blue
pcd2.translate([1.0,0.0,0.0])

print(pcd)
print(pcd2)

points = np.asarray(pcd.points)
points2 = np.asarray(pcd2.points)
print(points)
print(points2)

o3d.visualization.draw_geometries([pcd,pcd2], window_name="PLY Comparison")

PointCloud with 1768 points.
PointCloud with 1768 points.
[[  4.34101534  -1.97590983   8.94548512]
 [  6.54583645  10.24704552   2.88442421]
 [  4.26116467  -0.91624063   9.97420979]
 ...
 [-14.37703228  10.57718468   7.03714371]
 [ -3.41339111   4.91355276  12.40224266]
 [-14.48631001  10.56289577   6.95793295]]
[[  5.34101534  -1.97590983   8.94548512]
 [  7.54583645  10.24704552   2.88442421]
 [  5.26116467  -0.91624063   9.97420979]
 ...
 [-13.37703228  10.57718468   7.03714371]
 [ -2.41339111   4.91355276  12.40224266]
 [-13.48631001  10.56289577   6.95793295]]


## Training Setup

In [19]:
gaussian.training_setup(opt_args)